## Stage 2: CoT SFT Training
Uses chain-of-thought data generated by o4-mini, filtered to correct answers only.

In [ ]:
# Install required packages from offline source
!pip install -q --no-index --find-links /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages datasets trl --ignore-installed

In [ ]:
import datasets
import trl
print('datasets:', datasets.__version__)
print('trl:', trl.__version__)

In [ ]:
import os, sys, stat, shutil, gc, zipfile, json
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
import types

# --- Kaggle / Triton Environment Fixes (same as Stage 1) ---
def purermsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                   group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = purermsnorm_fn

src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell'
dst = '/tmp/ptxas-blackwell'
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    import importlib.util
    spec = importlib.util.find_spec('triton.backends.nvidia')
    real_nv_file = spec.origin if spec and spec.origin else None
    if real_nv_file:
        real_nv_dir = os.path.dirname(real_nv_file)
        src_bin = os.path.join(real_nv_dir, 'bin')
        if os.path.isdir(src_bin):
            dst_bin = '/tmp/triton_nvidia_bin'
            shutil.rmtree(dst_bin, ignore_errors=True)
            shutil.copytree(src_bin, dst_bin)
            for f in os.listdir(dst_bin):
                fp = os.path.join(dst_bin, f)
                if os.path.isfile(fp):
                    os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
            nv_backend.__file__ = os.path.join(dst_bin, '__init__.py')
            os.environ['TRITON_PTXAS_PATH'] = dst
            print('[INFO] Triton PTXAS fix applied.')

# --- Hyperparameters ---
LORA_RANK    = 32
MAX_SEQ_LEN  = 2048   # longer than Stage 1 (1024) to fit CoT reasoning
NUM_EPOCHS   = 1
GRAD_ACCUM   = 4
LR           = 2e-4
OUTPUT_DIR   = '/kaggle/working/adapter'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Data Loading — CoT JSONL

In [ ]:
COT_PATH = '/kaggle/input/nemotron-cot-stage2/train_cot.jsonl'

records = []
with open(COT_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f'Loaded {len(records)} CoT samples')
print('Example keys:', list(records[0].keys()))

## Model & Tokenizer Download

In [ ]:
MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Tokenizer loaded.')

## Format Training Text with CoT Reasoning

In [ ]:
def build_training_text(record):
    prompt    = record['prompt']
    reasoning = record['reasoning']
    answer    = record['answer']

    user_msg      = prompt + '\nPut your final answer inside \\boxed{}.'
    # Assistant shows full reasoning inside <think>, then the boxed answer
    assistant_msg = f'<think>{reasoning}</think>\\boxed{{{answer}}}'

    try:
        messages = [
            {'role': 'user',      'content': user_msg},
            {'role': 'assistant', 'content': assistant_msg},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    except Exception:
        text = (
            f'<|im_start|>user\n{user_msg}<|im_end|>\n'
            f'<|im_start|>assistant\n{assistant_msg}<|im_end|>'
        )
    return {'text': text}

hf_dataset = Dataset.from_list(records)
hf_dataset = hf_dataset.map(build_training_text, remove_columns=hf_dataset.column_names)

print(f'Dataset formatted. Total: {len(hf_dataset)} samples')
print('Example (first 600 chars):')
print(hf_dataset[0]['text'][:600])

## Model Loading & LoRA Setup

In [ ]:
for _mod_name in [
    'cutlass', 'cutlass.cute',
    'mamba_ssm.modules.mamba3',
    'mamba_ssm.ops.cute',
    'mamba_ssm.ops.cute.mamba3',
    'mamba_ssm.ops.cute.mamba3.mamba3_step_fn',
]:
    sys.modules[_mod_name] = types.ModuleType(_mod_name)
sys.modules['mamba_ssm.modules.mamba3'].Mamba3 = None

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
print(f'Model loaded. Vocab size: {len(tokenizer)}')

for name, mod in sys.modules.items():
    if 'modeling_nemotron_h' in name:
        mod.is_fast_path_available = False
        print(f'Patched {name}: is_fast_path_available = False')

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules='all-linear',
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Training

In [ ]:
import triton.backends.nvidia.compiler as nv_compiler
os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = '/tmp/ptxas-blackwell'
nv_compiler.get_ptxas_version = lambda arch: '12.0'

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=5,
    bf16=True,
    max_grad_norm=1.0,
    optim='adamw_torch',
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    save_strategy='no',
    report_to='none',
    dataset_text_field='text',
    max_length=MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': True},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=training_args,
)

print('Starting Stage 2 CoT training...')
trainer.train()

## Save & Package Submission

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
print(f'Adapter saved to {OUTPUT_DIR}:')
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f} ({size/1024:.1f} KB)')

zip_path = '/kaggle/working/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)

print(f'\nCreated {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)')
with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f'Contents: {names}')
    assert 'adapter_config.json' in names
    print('submission.zip ready to submit!')